# 🐱🚗 FarmTech Solutions — Ir Além 2 · Parte 3: Comparação final

**FIAP · Fase 6 · PBL — "Ir Além", Opção 2 (Transfer Learning, Fine Tuning e Segmentação)**

Integrantes: Victor Vergara Koshikene (RM 568813) · Leonardo Lima de Queiroz (RM 573539) · Vinicius Prado Speria
(RM 572741) · Leonardo Ribeiro de Souza (RM 571534) · Matheus Costa Manzoli Machado (RM 571814)

---

## Hipóteses

- **H1:** uma rede grande pré-treinada na ImageNet (EfficientNetB0 com fine tuning) classifica melhor que a CNN do zero?
- **H2:** pré-segmentar o objeto de interesse, apagando o fundo, facilita a classificação?

## Desenho experimental

| Classificador | Imagem original (O) | Imagem segmentada (S) |
|---|---|---|
| CNN do zero | treino O → teste O | treino S → teste S |
| Transfer Learning (EfficientNetB0) | treino O → teste O | treino S → teste S |

- Cada célula **treina e testa na mesma condição**. Um modelo treinado com fundo e testado sem fundo (ou o
  contrário) mede a mudança de distribuição, não o efeito da segmentação; isso aparece à parte, na Seção 6.
- Mesmos splits, mesma entrada (224×224), mesma augmentation e 5 sementes por célula.
- Conclusões no **teste ampliado** (52 imagens nunca vistas); o teste original (8) é reportado para comparar
  com a Entrega 2.
- **Ablações:** recorte pela caixa da máscara (C), máscara perfeita do COCO (G, "oráculo"), classificador que só
  vê a área da máscara e estratos com/sem pessoa na cena.

## Sumário

1. Configuração do ambiente
2. Dados das quatro condições
3. Treino na condição segmentada
4. Treino das ablações (recorte e oráculo)
5. Matriz 2×2
6. Mudança de distribuição (treino e teste em condições diferentes)
7. Ablações: o que explica o efeito da segmentação?
8. Para onde os modelos olham (CAM)
9. Latência do pipeline completo
10. Conclusões, limitações e próximos passos

## 1. Configuração do ambiente

Mesma lógica do notebook 01 (backend do Keras escolhido antes do import). O `ultralytics` é usado só para medir
a latência do pipeline completo (segmentação + classificação) na Seção 9. Requer os notebooks 01 e 02 executados
antes: aqui são lidos os pesos da condição O e as imagens segmentadas.

In [ ]:
# ---------------- Configuração do ambiente ----------------
import os, shutil, subprocess, sys, time, json
from pathlib import Path

NO_COLAB = "google.colab" in sys.modules
URL_REPO = "https://github.com/koshivv/fiap-farmtech_solutions-fase06.git"

if NO_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")                              # dados e pesos persistem entre sessões
    REPO_DIR = Path("/content/fiap-farmtech_solutions-fase06")
    if (REPO_DIR / "ir_alem_opcao2").exists():                 # sessão já tinha o clone: traz correções
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "-q", "--ff-only"], check=False)
    else:                                                      # main após o merge do PR; antes, o ramo do Ir Além
        shutil.rmtree(REPO_DIR, ignore_errors=True)
        for ramo in (os.environ.get("FARMTECH_RAMO", "main"), "ir-alem-opcao2"):
            subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", ramo, URL_REPO, str(REPO_DIR)], check=False)
            if (REPO_DIR / "ir_alem_opcao2").exists():
                break
            shutil.rmtree(REPO_DIR, ignore_errors=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.4.175", "pycocotools"], check=True)
else:                                                          # local: sobe até achar a raiz do repositório
    REPO_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "ir_alem_opcao2" / "src").is_dir())

sys.path.insert(0, str(REPO_DIR / "ir_alem_opcao2"))
from src import ambiente
BACKEND = ambiente.preparar_backend()

import keras
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from src import config, dados, metricas, modelos, segmentacao, treino, visualizacao as vis

config.garantir_pastas()
vis.aplicar_estilo()
AMBIENTE = ambiente.descrever_ambiente()
AMBIENTE["dispositivo_keras"] = ambiente.dispositivo_keras()
for chave, valor in AMBIENTE.items():
    print(f"{chave:18s}: {valor}")

## 2. Dados das quatro condições

In [ ]:
# ---------------- Tabelas e matrizes de entrada ----------------
coco = dados.carregar_coco()
tabela = dados.tabela_imagens(coco)
unica = tabela[tabela.grupo != "mista"]
TAB = {c: unica[unica.conjunto == c].reset_index(drop=True) for c in dados.CONJUNTOS}
segmentos = pd.read_csv(config.RESULTADOS_DIR / "segmentacao_imagens.csv")
ESCOLHA = json.loads((config.RESULTADOS_DIR / "segmentador_escolhido.json").read_text())
print("Segmentador (escolhido no notebook 02):", ESCOLHA)

# Falha cedo se o notebook 01 não rodou neste mesmo diretório de dados (evita horas de treino perdidas)
faltando = [n for n in ("predicoes_O.csv", "historicos_O.csv", "execucoes_O.csv", "ambiente_01.json")
            if not (config.RESULTADOS_DIR / n).exists()]
faltando += [f"{t}_O_s{s}" for t in ("cnn", "tl") for s in config.SEEDS if not treino.caminho_pesos(f"{t}_O_s{s}").exists()]
assert not faltando, f"Rode o notebook 01 antes (faltam: {faltando})"
AMBIENTE_01 = json.loads((config.RESULTADOS_DIR / "ambiente_01.json").read_text(encoding="utf-8"))
if (AMBIENTE_01.get("keras_backend"), AMBIENTE_01.get("dispositivo_keras")) != (BACKEND, AMBIENTE["dispositivo_keras"]):
    print("⚠️  O notebook 01 rodou com", AMBIENTE_01.get("keras_backend"), AMBIENTE_01.get("dispositivo_keras"),
          "e este com", BACKEND, AMBIENTE["dispositivo_keras"], ": tempos de treino não são comparáveis entre O e S.")

X = {cond: dados.carregar_condicao(TAB, cond) for cond in ("O", "S", "C", "G")}        # avaliação (224x224)
TREINO = {cond: dados.imagens_treino(TAB["treino"], cond) for cond in ("S", "C", "G")}  # treino (augmentation)
NOMES_COND = {"O": "original", "S": "segmentada", "C": "recorte", "G": "oráculo (GT)"}
permitidas = TAB["treino"][TAB["treino"].permite_derivados]    # licença permite publicar versões alteradas
exemplos = [permitidas[permitidas.grupo == g].index[0] for g in config.CLASSES]   # um gato e um carro do treino
fig, eixos = plt.subplots(2, 4, figsize=(15, 7.5))
for linha, indice in zip(eixos, exemplos):
    for ax, cond in zip(linha, NOMES_COND):
        ax.imshow(X[cond]["treino"][indice].astype("uint8"))
        ax.set_title(f"{NOMES_COND[cond]} ({cond})")
        ax.axis("off")
fig.suptitle("A mesma imagem de treino nas quatro condições (entrada 224×224)", fontsize=13)
fig.tight_layout()
plt.show()

## 3. Treino na condição segmentada

Mesmo protocolo do notebook 01 (CNN: 100 épocas; EfficientNetB0: 30 + 30 épocas), agora com treino, validação e
testes na condição S. Imagens em que a segmentação falhou entram como originais (fallback), exatamente como
aconteceria em produção.

In [ ]:
# ---------------- Treino S (e leitura dos resultados O do notebook 01) ----------------
inicio = time.perf_counter()
execucoes = []
for seed in config.SEEDS:
    for tipo in ("cnn", "tl"):
        execucoes.append(treino.executar(tipo, "S", seed, TREINO["S"], X["S"], TAB))
print(f"⏱️  Condição S: {time.perf_counter() - inicio:.0f} s")

## 4. Treino das ablações (recorte e oráculo)

- **C (recorte):** a imagem segmentada é recortada pela caixa da máscara antes do redimensionamento. Se C ganhar de
  S, parte do efeito vem de **ampliar** o objeto (mais pixels para ele), e não de apagar o fundo.
- **G (oráculo):** fundo apagado com a máscara humana do COCO. Estima quanto a H2 ganharia com uma segmentação
  próxima da perfeita; não é um pipeline automático nem um teto garantido (os polígonos do COCO são grosseiros).

In [ ]:
# ---------------- Treino C e G ----------------
inicio = time.perf_counter()
for condicao in ("C", "G"):
    for seed in config.SEEDS:
        for tipo in ("cnn", "tl"):
            execucoes.append(treino.executar(tipo, condicao, seed, TREINO[condicao], X[condicao], TAB))
print(f"⏱️  Ablações: {time.perf_counter() - inicio:.0f} s")

predicoes = pd.concat([pd.read_csv(config.RESULTADOS_DIR / "predicoes_O.csv")]
                      + [r["predicoes"] for r in execucoes], ignore_index=True)
historicos = pd.concat([pd.read_csv(config.RESULTADOS_DIR / "historicos_O.csv")]
                       + [r["historico"] for r in execucoes], ignore_index=True)
resumo_exec = pd.concat([pd.read_csv(config.RESULTADOS_DIR / "execucoes_O.csv")]
                        + [r["execucao"] for r in execucoes], ignore_index=True)
predicoes.to_csv(config.RESULTADOS_DIR / "predicoes_todas.csv", index=False)
historicos.to_csv(config.RESULTADOS_DIR / "historicos_todos.csv", index=False)
resumo_exec.to_csv(config.RESULTADOS_DIR / "execucoes_todas.csv", index=False)

## 5. Matriz 2×2

### 5.1 Tabela principal (média ± desvio padrão em 5 sementes)

In [ ]:
# ---------------- Matriz 2x2 ----------------
finais = predicoes[(predicoes.fase == "final") & (predicoes.condicao_treino == predicoes.condicao_teste)]
por_semente = metricas.metricas_por_semente(finais, ["conjunto", "modelo", "condicao_treino"])
por_semente.to_csv(config.RESULTADOS_DIR / "metricas_por_semente_todas.csv", index=False)
COLUNAS = ["acuracia", "f1_macro", "log_loss", "brier", "p_correta"]
NOME_MODELO = {"cnn": "CNN do zero", "tl": "Transfer Learning (EfficientNetB0)"}

matriz = []
for conjunto in ("teste_ampliado", "teste_original"):
    sel = por_semente[(por_semente.conjunto == conjunto) & por_semente.condicao_treino.isin(["O", "S"])]
    for metrica in ("acuracia", "log_loss"):
        tab = metricas.media_desvio(sel, ["modelo", "condicao_treino"], [metrica])[metrica].unstack()
        tab = tab.rename(index=NOME_MODELO, columns={"O": "Imagem original", "S": "Imagem segmentada"})
        tab.insert(0, "métrica", metrica)
        tab.insert(0, "conjunto", conjunto)
        matriz.append(tab)
matriz = pd.concat(matriz)
matriz.to_csv(config.RESULTADOS_DIR / "matriz_2x2.csv")
display(matriz)

In [ ]:
# ---------------- Figura da matriz 2x2 (teste ampliado) ----------------
amp = por_semente[(por_semente.conjunto == "teste_ampliado") & por_semente.condicao_treino.isin(["O", "S"])]
fig, eixos = plt.subplots(1, 3, figsize=(16, 4.3))
for ax, (metrica, titulo) in zip(eixos, [("acuracia", "Acurácia (↑)"), ("log_loss", "Log-loss (↓)"),
                                         ("p_correta", "P(classe correta) (↑)")]):
    resumo = amp.groupby(["modelo", "condicao_treino"])[metrica].agg(["mean", "std"]).reset_index()
    for j, cond in enumerate(("O", "S")):
        r = resumo[resumo.condicao_treino == cond]
        barras = ax.bar(np.arange(2) + (j - 0.5) * 0.38, r["mean"], 0.38, yerr=r["std"], capsize=4,
                        color=vis.CORES["original" if cond == "O" else "segmentada"], label=NOMES_COND[cond])
        ax.bar_label(barras, fmt="%.3f", fontsize=8, padding=2)
    ax.set_xticks(range(2), ["CNN do zero", "EfficientNetB0 (TL)"])
    ax.set_title(titulo)
eixos[0].set_ylim(0, 1.12)
eixos[0].legend(loc="lower right")
fig.suptitle("Matriz 2×2 no teste ampliado (52 imagens, média ± desvio de 5 sementes)", fontsize=13)
fig.tight_layout()
vis.salvar(fig, "matriz_2x2.png")
plt.show()

### 5.2 Precisão, recall e F1 por classe (teste ampliado)

In [ ]:
COLUNAS_CLASSE = [f"{m}_{c}" for c in config.CLASSES for m in ("precisao", "recall", "f1")]
display(metricas.media_desvio(amp, ["modelo", "condicao_treino"], COLUNAS_CLASSE).rename(index=NOME_MODELO))

### 5.3 Matrizes de confusão (soma das 5 sementes, teste ampliado)

In [ ]:
paineis = []
for tipo in ("cnn", "tl"):
    for cond in ("O", "S"):
        sel = finais[(finais.conjunto == "teste_ampliado") & (finais.modelo == tipo) & (finais.condicao_treino == cond)]
        paineis.append((f"{NOME_MODELO[tipo].split(' (')[0]} · {NOMES_COND[cond]}", metricas.matriz_confusao(sel)))
fig = vis.matrizes_confusao(paineis)
vis.salvar(fig, "matriz_2x2_confusao.png")
plt.show()

### 5.4 Curvas de treino na condição segmentada

In [ ]:
fig = vis.curvas_treino(historicos[(historicos.modelo == "cnn") & (historicos.condicao == "S")],
                        "CNN do zero — condição segmentada (5 sementes)")
vis.salvar(fig, "h2_curvas_cnn_S.png")
plt.show()
fig = vis.curvas_treino(historicos[(historicos.modelo == "tl") & (historicos.condicao == "S")],
                        "EfficientNetB0 — condição segmentada (fase 1 + fine tuning)", inicio_fase2=config.EPOCAS_TL_FASE1)
vis.salvar(fig, "h2_curvas_tl_S.png")
plt.show()

### 5.5 Testes pareados (consenso das 5 sementes, McNemar exato)

Compara duas configurações **nas mesmas imagens**: original × segmentada para cada modelo (H2) e TL × CNN em cada
condição (H1). No teste ampliado, o p-valor é ajustado por Holm dentro de cada hipótese (2 comparações para a H1,
2 para a H2). No teste original (8 imagens), o teste é só descritivo: com tão poucas imagens, nenhuma diferença
menor que 6 discordâncias no mesmo sentido pode ser significativa.

In [ ]:
consenso = metricas.predicao_consenso(finais, ["conjunto", "modelo", "condicao_treino"])
def comparar(conjunto, a, b):
    sel = consenso[consenso.conjunto == conjunto]
    ca = sel[(sel.modelo == a[0]) & (sel.condicao_treino == a[1])].set_index("arquivo").correto
    cb = sel[(sel.modelo == b[0]) & (sel.condicao_treino == b[1])].set_index("arquivo").correto.loc[ca.index]
    teste = metricas.mcnemar_exato(ca, cb)
    return {"conjunto": conjunto, "A": f"{a[0]}·{a[1]}", "B": f"{b[0]}·{b[1]}",
            "acurácia A": round(ca.mean(), 3), "IC95% A": metricas.ic_bootstrap(ca),
            "acurácia B": round(cb.mean(), 3), "IC95% B": metricas.ic_bootstrap(cb),
            "só A acerta": teste["so_A_acerta"], "só B acerta": teste["so_B_acerta"], "p-valor": round(teste["p_valor"], 4)}
pares = {"H2": [(("cnn", "S"), ("cnn", "O")), (("tl", "S"), ("tl", "O"))],
         "H1": [(("tl", "O"), ("cnn", "O")), (("tl", "S"), ("cnn", "S"))]}
linhas = []
for conjunto in ("teste_ampliado", "teste_original"):
    for hipotese, comparacoes in pares.items():
        bloco = [comparar(conjunto, a, b) | {"hipótese": hipotese} for a, b in comparacoes]
        ajustados = metricas.holm([t["p-valor"] for t in bloco])
        for t, p in zip(bloco, ajustados):
            t["p (Holm)"] = p if conjunto == "teste_ampliado" else "descritivo"
        linhas += bloco
testes = pd.DataFrame(linhas)
testes.to_csv(config.RESULTADOS_DIR / "testes_pareados.csv", index=False)
display(testes.set_index(["conjunto", "hipótese", "A", "B"]))

## 6. Mudança de distribuição (treino e teste em condições diferentes)

Por que o desenho treina e testa na mesma condição: abaixo, os modelos treinados em uma condição são avaliados na
outra, sem retreino. (Recalculado a cada execução, a partir dos pesos salvos.)

In [ ]:
# ---------------- Avaliação cruzada O <-> S ----------------
cruzadas = pd.concat([treino.avaliar_salvo(tipo, f"{tipo}_{origem}_s{seed}", X[destino], TAB, origem, destino, seed)
                      for tipo in ("cnn", "tl") for seed in config.SEEDS
                      for origem, destino in (("O", "S"), ("S", "O"))], ignore_index=True)
cruzadas.to_csv(config.RESULTADOS_DIR / "predicoes_cruzadas.csv", index=False)
todas = pd.concat([finais[finais.condicao_treino.isin(["O", "S"])], cruzadas], ignore_index=True)
cruz = metricas.metricas_por_semente(todas[todas.conjunto == "teste_ampliado"], ["modelo", "condicao_treino", "condicao_teste"])
tabela_cruz = metricas.media_desvio(cruz, ["modelo", "condicao_treino", "condicao_teste"], ["acuracia", "log_loss"])
tabela_cruz.to_csv(config.RESULTADOS_DIR / "mudanca_distribuicao.csv")
display(tabela_cruz.rename(index=NOME_MODELO))

## 7. Ablações: o que explica o efeito da segmentação?

In [ ]:
# ---------------- Condições O, S, C, G e referências ----------------
amp4 = por_semente[por_semente.conjunto == "teste_ampliado"]
ablacao = metricas.media_desvio(amp4, ["modelo", "condicao_treino"], ["acuracia", "log_loss", "p_correta"])
ablacao = ablacao.rename(index={**NOME_MODELO, **NOMES_COND})
ablacao.to_csv(config.RESULTADOS_DIR / "ablacoes.csv")
display(ablacao)

referencias = pd.DataFrame({
    "acurácia no teste ampliado": [
        pd.read_csv(config.RESULTADOS_DIR / "baseline_area.csv").set_index("conjunto")
          .loc["teste_ampliado", "acurácia só pela área"],
        pd.read_csv(config.RESULTADOS_DIR / "segmentacao_como_classificador.csv").set_index("conjunto")
          .loc["teste_ampliado", "acurácia implícita"]]},
    index=["classificador que só vê a fração visível (limiar do treino)", f"segmentador sozinho ({ESCOLHA['segmentador']})"])
display(referencias)

In [ ]:
# ---------------- Estrato "pessoa na cena" (teste ampliado, 5 sementes) ----------------
amp_pred = finais[(finais.conjunto == "teste_ampliado") & finais.condicao_treino.isin(["O", "S"])]
estrato = (amp_pred.groupby(["modelo", "condicao_treino", "grupo", "tem_pessoa"]).correto.mean()
           .unstack(["grupo", "tem_pessoa"]).round(3))
estrato.to_csv(config.RESULTADOS_DIR / "estrato_pessoa.csv")
display(estrato.rename(index={**NOME_MODELO, **NOMES_COND}))
print("Imagens por estrato:", TAB["teste_ampliado"].groupby(["grupo", "tem_pessoa"]).size().to_dict())

## 8. Para onde os modelos olham (CAM)

*Class Activation Map* da classe prevista (semente 42): regiões em vermelho pesaram mais na decisão. Como a cabeça é
GAP → Dense, o CAM é exato e não precisa de gradientes. Imagens do teste ampliado com licença que permite obras
derivadas, priorizando as que a CNN errou na condição original.

In [ ]:
# ---------------- CAM: original x segmentada ----------------
cons_cnn_O = consenso[(consenso.conjunto == "teste_ampliado") & (consenso.modelo == "cnn") & (consenso.condicao_treino == "O")]
erradas = set(cons_cnn_O[~cons_cnn_O.correto].arquivo)
candidatas = TAB["teste_ampliado"][TAB["teste_ampliado"].permite_derivados]
indices = []
for grupo in config.CLASSES:                                    # 2 gatos e 2 carros; erros da CNN primeiro
    opcoes = candidatas[candidatas.grupo == grupo]
    indices += sorted(opcoes.index, key=lambda i: (opcoes.arquivo[i] not in erradas, opcoes.arquivo[i]))[:2]

paineis_cam = [("cnn", "O"), ("cnn", "S"), ("tl", "O"), ("tl", "S")]
submodelos = {}
for tipo, cond in paineis_cam:
    keras.backend.clear_session()
    submodelos[(tipo, cond)] = modelos.modelo_cam(treino.carregar_modelo(tipo, f"{tipo}_{cond}_s{config.SEED}"))
fig, eixos = plt.subplots(len(indices), 4, figsize=(15, 3.9 * len(indices)), squeeze=False)
for linha, i in zip(eixos, indices):
    for ax, (tipo, cond) in zip(linha, paineis_cam):
        imagem = X[cond]["teste_ampliado"][i]
        cam, classe, prob = vis.calcular_cam(submodelos[(tipo, cond)], imagem)
        vis.sobrepor_cam(ax, imagem, cam, f"{'CNN' if tipo == 'cnn' else 'TL'} · {NOMES_COND[cond]}\n"
                                          f"→ {config.CLASSES[classe]} ({prob[classe]:.2f})")
    linha[0].set_ylabel(f"{TAB['teste_ampliado'].grupo[i]} · {TAB['teste_ampliado'].arquivo[i]}", fontsize=9)
fig.tight_layout()
vis.salvar(fig, "cam_original_vs_segmentada.png")
plt.show()

## 9. Latência do pipeline completo

Por imagem, uma de cada vez, após aquecimento: **O** = redimensionar + classificar; **S** = segmentar + apagar o
fundo + redimensionar + classificar.

In [ ]:
# ---------------- Latência ----------------
seg = segmentacao.Segmentador(ESCOLHA["segmentador"],
                              config.CLASSES_SEG_ESTRITAS if ESCOLHA["classes"] == "estritas" else config.CLASSES_SEG_AMPLAS)
imagens_lat = [dados.abrir_imagem(config.IMAGENS_DIR / a) for a in TAB["teste_ampliado"].arquivo[:18]]

def pipeline_O(modelo):
    return lambda img: modelo.predict_on_batch(dados.redimensionar(img)[None].astype("float32"))

def pipeline_S(modelo):
    def prever(img):
        r = segmentacao.segmentar(seg, img)
        base = segmentacao.aplicar_mascara(np.asarray(img), r.mascara) if r.mascara is not None else np.asarray(img)
        return modelo.predict_on_batch(dados.redimensionar(base)[None].astype("float32"))
    return prever

linhas = []
for tipo in ("cnn", "tl"):
    for cond, fabrica in (("O", pipeline_O), ("S", pipeline_S)):
        keras.backend.clear_session()
        modelo = treino.carregar_modelo(tipo, f"{tipo}_{cond}_s{config.SEED}")
        media, desvio = metricas.medir_latencia(fabrica(modelo), imagens_lat, repeticoes=3)
        linhas.append({"modelo": NOME_MODELO[tipo], "pipeline": NOMES_COND[cond], "ms/img": round(media, 1),
                       "desvio": round(desvio, 1)})
latencia = pd.DataFrame(linhas)
latencia["dispositivo"] = f"classificador: {AMBIENTE['dispositivo_keras']} ({BACKEND}); segmentador: " + \
                          (AMBIENTE.get("gpu_nome", "CPU") if AMBIENTE.get("gpu_torch") else "CPU")
latencia.to_csv(config.RESULTADOS_DIR / "latencia_pipeline.csv", index=False)
display(latencia)
tempo = resumo_exec.groupby(["modelo", "condicao"]).tempo_treino_s.mean().unstack().round(1)
display(tempo.rename(index=NOME_MODELO))

In [ ]:
# ---------------- Exporta os CSVs para o repositório ----------------
ambiente.exportar(["predicoes_todas.csv", "historicos_todos.csv", "execucoes_todas.csv",
                   "metricas_por_semente_todas.csv", "matriz_2x2.csv", "testes_pareados.csv", "predicoes_cruzadas.csv",
                   "mudanca_distribuicao.csv", "ablacoes.csv", "estrato_pessoa.csv", "latencia_pipeline.csv"],
                  AMBIENTE, "ambiente_03.json")

## 10. Conclusões, limitações e próximos passos

<!-- CONCLUSAO_FINAL -->
*(preenchida após a execução de referência)*